# ??? *Capsicum annuum* (Samanda? Biberi) Kapsaisin Biyosentez Yola?? ve Pun1 Enzim Analizi

Bu ?al??ma, **NCBI RefSeq** (*Capsicum annuum* `GCF_002878395.1` - UCD10Xv1.1) tam referans genomu ?zerinden kapsaisin ve dihidrokapsaisin biyosentezinden sorumlu gen ve protein dizilerini izole etmek, **BAHD a?iltransferaz (`HXXXD` / `DFGWG`)** katalitik motifleri ile ger?ek **Kapsaisin Sentaz (Pun1/AT3)** adaylar?n? do?rulamak ve **ColabFold / AlphaFold2** 3B yap? tahminleri i?in girdi setlerini haz?rlamak amac?yla geli?tirilmi?tir.

---
## 1. Gerekli Ara?lar?n ve K?t?phanelerin Kurulumu
Genom verilerinin indirilmesi i?in **NCBI Datasets CLI** ara?lar? ve sekans analizi i?in **Biopython** k?t?phanesi kurulur.


In [1]:
# Biopython kurulumu ve NCBI resmi veri indirme araçlarının (datasets CLI) indirilmesi
!pip install -q biopython
!wget -q https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets -O /usr/local/bin/datasets
!wget -q https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/dataformat -O /usr/local/bin/dataformat
!chmod +x /usr/local/bin/datasets /usr/local/bin/dataformat

# Kurulumu doğrulama
!datasets --version

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 7.3 MB/s eta 0:00:00
datasets version: 18.39.0


## 2. *Capsicum annuum* Referans Genomunun ?ndirilmesi
`GCF_002878395.1` (UCD10Xv1.1 RefSeq) eri?im numaral? tam genom (`fna`), anotasyon (`gff3`), kodlayan diziler (`cds`) ve protein (`faa`) paketleri indirilir.


In [2]:
import os
import glob

# Referans genom erişim numarası (Capsicum annuum - UCD10Xv1.1 RefSeq)
ACCESSION = "GCF_002878395.1"
OUTPUT_DIR = "/content/biber_genomu"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"{ACCESSION} referans tam genomu indiriliyor (yaklaşık 1-2 dakika sürebilir)...")

# Tam genom (fna), anotasyon (gff3), protein (faa) ve CDS dosyalarını indirme
!datasets download genome accession {ACCESSION} \
    --include genome,gff3,protein,cds \
    --filename {OUTPUT_DIR}/capsicum_annuum.zip

# İndirilen arşivi açma
!unzip -q -o {OUTPUT_DIR}/capsicum_annuum.zip -d {OUTPUT_DIR}

# İndirilen dosyaların yollarını listeleme
data_path = f"{OUTPUT_DIR}/ncbi_dataset/data/{ACCESSION}"
for file in os.listdir(data_path):
    full_path = os.path.join(data_path, file)
    size_mb = os.path.getsize(full_path) / (1024 * 1024)
    print(f"Dosya: {file} ({size_mb:.2f} MB)")

GCF_002878395.1 referans tam genomu indiriliyor (yakla??k 1-2 dakika s?rebilir)...
Downloading: /content/biber_genomu/capsicum_annuum.zip    1.05GB valid data package
Validating package files [================================================] 100% 8/8
Dosya: protein.faa (25.29 MB)
Dosya: cds_from_genomic.fna (79.73 MB)
Dosya: GCF_002878395.1_UCD10Xv1.1_genomic.fna (3112.06 MB)
Dosya: genomic.gff (287.04 MB)


## 3. Kapsaisin Biyosentez Yola?? ?li?kili CDS ve Protein Dizilerinin Filtrelenmesi
Tam genom i?erisindeki on binlerce dizi aras?ndan kapsaisin biyosentez yola?? enzimleriyle ili?kili anahtar kelimeleri i?eren **CDS** ve **protein** kay?tlar? ay?klan?r.


In [3]:
from Bio import SeqIO
import glob
import os

ACCESSION = "GCF_002878395.1"
VERI_KLASORU = f"/content/biber_genomu/ncbi_dataset/data/{ACCESSION}"
CIKTI_KLASORU = "/content/kapsaisin_genleri"
os.makedirs(CIKTI_KLASORU, exist_ok=True)

# RefSeq anotasyonlarında kapsaisin yolağı enzimlerini yakalayacak anahtar kelimeler
anahtar_kelimeler = [
    "capsaicin", "pun1", "at3",
    "putative aminotransferase", "vanillin aminotransferase",
    "ketoacyl-acp synthase", "3-oxoacyl-[acyl-carrier-protein] synthase",
    "branched-chain-amino-acid aminotransferase",
    "caffeic acid 3-o-methyltransferase",
    "acyl-[acyl-carrier-protein] thioesterase",
    "phenylalanine ammonia-lyase",
    "cinnamate 4-hydroxylase",
    "transcription factor myb31"
]

# 1. CDS (Kodlayan DNA dizileri) dosyasından filtreleme
cds_dosyalari = glob.glob(f"{VERI_KLASORU}/*cds_from_genomic.fna")
if cds_dosyalari:
    secilen_cds = []
    for record in SeqIO.parse(cds_dosyalari[0], "fasta"):
        tanim_kucuk = record.description.lower()
        if any(kelime in tanim_kucuk for kelime in anahtar_kelimeler):
            secilen_cds.append(record)

    cds_cikti = f"{CIKTI_KLASORU}/referans_genom_kapsaisin_CDS.fasta"
    SeqIO.write(secilen_cds, cds_cikti, "fasta")
    print(f"Tam genomdan {len(secilen_cds)} adet ilişkili CDS dizisi ayıklandı -> {cds_cikti}")

# 2. Protein (.faa) dosyasından filtreleme
protein_dosyalari = glob.glob(f"{VERI_KLASORU}/*.faa")
if protein_dosyalari:
    secilen_proteinler = []
    for record in SeqIO.parse(protein_dosyalari[0], "fasta"):
        tanim_kucuk = record.description.lower()
        if any(kelime in tanim_kucuk for kelime in anahtar_kelimeler):
            secilen_proteinler.append(record)

    prot_cikti = f"{CIKTI_KLASORU}/referans_genom_kapsaisin_Protein.faa"
    SeqIO.write(secilen_proteinler, prot_cikti, "fasta")
    print(f"Tam genomdan {len(secilen_proteinler)} adet ilişkili Protein dizisi ayıklandı -> {prot_cikti}")

Tam genomdan 993 adet ilişkili CDS dizisi ayıklandı -> /content/kapsaisin_genleri/referans_genom_kapsaisin_CDS.fasta
Tam genomdan 993 adet ilişkili Protein dizisi ayıklandı -> /content/kapsaisin_genleri/referans_genom_kapsaisin_Protein.faa


## 4. Katalitik Motif (`HXXXD` & `DFGWG`) ve Sekans Hizalamas? ile Pun1 (Kapsaisin Sentaz) ?zolasyonu
Metin tabanl? aramadan gelen 993 aday protein i?erisindeki yalanc? pozitifler (`At3g`, `GPAT3`, `STAT3` vb.) elenir. Do?rulanm?? aktif *Capsicum annuum* Kapsaisin Sentaz referans? (**`AAV66311.1`**) ile global sekans hizalamas? (`PairwiseAligner`) ve BAHD ailesi katalitik motif kontrol? yap?larak ?ekirdek biyosentez enzimleri izole edilir.


In [4]:
from Bio import SeqIO, Entrez
from Bio.Align import PairwiseAligner
import pandas as pd
import re
import os

Entrez.email = "arastirmaci@universite.edu.tr"

GIRIS_DOSYASI = "/content/kapsaisin_genleri/referans_genom_kapsaisin_Protein.faa"
CIKTI_KLASORU = "/content/kapsaisin_genleri/izole_hatay_seti"
os.makedirs(CIKTI_KLASORU, exist_ok=True)

# 1. NCBI'dan kanonik (doğrulanmış aktif) Capsicum annuum Kapsaisin Sentaz (Pun1) proteinini çekme
print("1. Doğrulanmış aktif Pun1 (AAV66311.1) referans proteini NCBI'dan alınıyor...")
handle = Entrez.efetch(db="protein", id="AAV66311.1", rettype="fasta", retmode="text")
referans_pun1 = SeqIO.read(handle, "fasta")
handle.close()
print(f"   Referans Pun1 yüklendi: {referans_pun1.id} ({len(referans_pun1.seq)} aa)\n")

# 2. 993 proteini okuma ve 'At3g' gibi yalancı pozitiflerden temizleme
tum_proteinler = list(SeqIO.parse(GIRIS_DOSYASI, "fasta"))
print(f"2. Başlangıç protein sayısı: {len(tum_proteinler)}")

# Katalitik motif düzenli ifadeleri (BAHD Açiltransferaz / Kapsaisin Sentaz imzası)
motif_HXXXD = re.compile(r"H[A-Z]{3}D")
motif_DFGWG = re.compile(r"DFGWG|DFGFG")

aligner = PairwiseAligner()
aligner.mode = "global"

pun1_adaylari = []
cekirdek_yolak_adaylari = {}

# Samandağ biberinin Kapsaisin + Dihidrokapsaisin (550-1200 ppm) sentezini yöneten 7 anahtar enzim
hedef_enzim_kategorileri = {
    "1_Pun1_Kapsaisin_Sentaz": ["capsaicin synthase", "acyltransferase", "pun1"],
    "2_pAMT_Vanilin_Aminotransferaz": ["gamma-aminobutyrate transaminase", "vanillin aminotransferase", "putative aminotransferase"],
    "3_KasI_Ketoacil_ACP_Sentaz": ["3-oxoacyl-[acyl-carrier-protein] synthase i", "ketoacyl-acp synthase"],
    "4_FatA_Acil_ACP_Tiyoesteraz": ["acyl-[acyl-carrier-protein] thioesterase"],
    "5_BCAT_Dalli_Zincir_Aminotransferaz": ["branched-chain-amino-acid aminotransferase"],
    "6_COMT_Kafeik_Asit_O_Metiltransferaz": ["caffeic acid 3-o-methyltransferase"],
    "7_PAL_Fenilalanin_Amonyak_Liyaz": ["phenylalanine ammonia-lyase"]
}

for rec in tum_proteinler:
    tanim = rec.description.lower()
    seq_str = str(rec.seq).upper()
    uzunluk = len(seq_str)

    # 'at3g', 'gpat3', 'lpat3', 'cat3', 'stat3', 'low quality' gibi yalancı pozitifleri ele
    if any(x in tanim for x in ["at3g", "gpat3", "lpat3", "cat3", "stat3", "vat3", "mat3", "low quality"]):
        continue

    # A) Gerçek Kapsaisin Sentaz (Pun1) Adaylarını Motif + Uzunluk + Hizalama ile Yakala
    if (380 <= uzunluk <= 480) and motif_HXXXD.search(seq_str) and motif_DFGWG.search(seq_str):
        skor = aligner.score(str(referans_pun1.seq), seq_str)
        benzerlik_yuzdesi = (skor / max(len(referans_pun1.seq), uzunluk)) * 100

        if benzerlik_yuzdesi > 55:  # BAHD ailesi içinden Pun1'e en yakın olanlar
            hxxxd_eslesme = motif_HXXXD.search(seq_str)
            dfgwg_eslesme = motif_DFGWG.search(seq_str)
            pun1_adaylari.append({
                "Record": rec,
                "Protein_ID": rec.id,
                "Uzunluk_aa": uzunluk,
                "Pun1_Benzerlik (%)": round(benzerlik_yuzdesi, 2),
                "HXXXD_Motifi": f"{hxxxd_eslesme.group(0)} (Poz: {hxxxd_eslesme.start()+1})",
                "DFGWG_Motifi": f"{dfgwg_eslesme.group(0)} (Poz: {dfgwg_eslesme.start()+1})",
                "Açıklama": rec.description
            })

    # B) Samandağ Biberi Çekirdek Yolak Enzimlerinden En Temsili (Kanonik) Olanları Seç
    for kategori, kelimeler in hedef_enzim_kategorileri.items():
        if any(k in tanim for k in kelimeler) and "partial" not in tanim:
            # Her kategoriden en yüksek kaliteli / kanonik uzunluktaki tek temsilciyi tut
            if kategori not in cekirdek_yolak_adaylari:
                cekirdek_yolak_adaylari[kategori] = rec

# Pun1 adaylarını benzerlik yüzdesine göre büyükten küçüğe sırala
pun1_adaylari = sorted(pun1_adaylari, key=lambda x: x["Pun1_Benzerlik (%)"], reverse=True)

# En yüksek eşleşmeli Pun1'i çekirdek sete 1 numara olarak ekle
if pun1_adaylari:
    cekirdek_yolak_adaylari["1_Pun1_Kapsaisin_Sentaz"] = pun1_adaylari[0]["Record"]

# 3. Sonuçları Dosyalara Yazdırma
# Dosya 1: İzole edilmiş en güçlü Pun1 (Kapsaisin Sentaz) adayları
en_iyi_pun1_kayitlari = [x["Record"] for x in pun1_adaylari[:5]]
pun1_cikti = f"{CIKTI_KLASORU}/izole_Pun1_Kapsaisin_Sentaz_Adaylari.faa"
SeqIO.write(en_iyi_pun1_kayitlari, pun1_cikti, "fasta")

# Dosya 2: 993 proteinden damıtılmış Çekirdek Samandağ Kapsaisin Yolağı Seti (Her genden 1 kanonik protein)
cekirdek_kayitlar = []
for kat, r in sorted(cekirdek_yolak_adaylari.items()):
    r.description = f"[{kat}] {r.description}"
    cekirdek_kayitlar.append(r)

cekirdek_cikti = f"{CIKTI_KLASORU}/Samandag_Cekirdek_Kapsaisin_Yolagi_7_Protein.faa"
SeqIO.write(cekirdek_kayitlar, cekirdek_cikti, "fasta")

# Tablo raporu
df_pun1 = pd.DataFrame([{k: v for k, v in x.items() if k != "Record"} for x in pun1_adaylari[:10]])
df_pun1.to_csv(f"{CIKTI_KLASORU}/Pun1_Motif_ve_Benzerlik_Raporu.csv", index=False, encoding="utf-8-sig")

print(f"\n3. FİLTRELEME TAMAMLANDI!")
print(f"   -> 993 protein içinden {len(pun1_adaylari)} adet tam motifli (HXXXD + DFGWG) aktif açiltransferaz bulundu.")
print(f"   -> En yüksek eşleşmeli ilk 5 Kapsaisin Sentaz adayı kaydedildi: {pun1_cikti}")
print(f"   -> Yolak üzerindeki {len(cekirdek_kayitlar)} anahtar enzim tekil olarak izole edildi: {cekirdek_cikti}\n")
display(df_pun1)

1. Doğrulanmış aktif Pun1 (AAV66311.1) referans proteini NCBI'dan alınıyor...
   Referans Pun1 yüklendi: AAV66311.1 (440 aa)

2. Başlangıç protein sayısı: 993

3. FİLTRELEME TAMAMLANDI!
   -> 993 protein içinden 2 adet tam motifli (HXXXD + DFGWG) aktif açiltransferaz bulundu.
   -> En yüksek eşleşmeli ilk 5 Kapsaisin Sentaz adayı kaydedildi: /content/kapsaisin_genleri/izole_hatay_seti/izole_Pun1_Kapsaisin_Sentaz_Adaylari.faa
   -> Yolak üzerindeki 5 anahtar enzim tekil olarak izole edildi: /content/kapsaisin_genleri/izole_hatay_seti/Samandag_Cekirdek_Kapsaisin_Yolagi_7_Protein.faa



,Protein_ID,Uzunluk_aa,Pun1_Benzerlik (%),HXXXD_Motifi,DFGWG_Motifi,Açıklama
0,NP_001311698.1,440,99.55,HKIGD (Poz: 169),DFGWG (Poz: 384),NP_001311698.1 acyltransferase Pun1 [Capsicum ...
1,XP_016560262.2,459,80.61,HKIGD (Poz: 186),DFGWG (Poz: 403),XP_016560262.2 acyltransferase Pun1 [Capsicum ...


## 5. NCBI Veritaban?nda T?rkiye Kaynakl? *Capsicum annuum* Kay?tlar?n?n Sorgulanmas?
NCBI Entrez API kullan?larak T?rkiye / Hatay / Samanda? men?eli *Capsicum annuum* protein kay?tlar? taran?r.


In [5]:
# NCBI'da Türkiye menşeli Capsicum annuum gen/protein kayıtlarını sorgulama
sorgu_turkiye = 'Capsicum annuum[Organism] AND (Turkey[Country] OR Turkey[Title] OR Hatay[All Fields] OR Samandag[All Fields] OR Antakya[All Fields])'

handle = Entrez.esearch(db="protein", term=sorgu_turkiye, retmax=50)
tr_sonuc = Entrez.read(handle)
handle.close()

print(f"NCBI Protein veritabanında Türkiye kaynaklı Capsicum annuum kayıt sayısı: {tr_sonuc['Count']}")

if tr_sonuc["IdList"]:
    fetch_handle = Entrez.efetch(db="protein", id=tr_sonuc["IdList"], rettype="fasta", retmode="text")
    tr_proteinler = list(SeqIO.parse(fetch_handle, "fasta"))
    fetch_handle.close()

    tr_dosya = f"{CIKTI_KLASORU}/NCBI_Turkiye_Biber_Proteinleri.faa"
    SeqIO.write(tr_proteinler, tr_dosya, "fasta")
    print(f"Türkiye kaynaklı {len(tr_proteinler)} protein dizisi '{tr_dosya}' dosyasına kaydedildi.\n")

    for p in tr_proteinler[:10]:
        print(f" - {p.id}: {p.description[:90]}... ({len(p.seq)} aa)")

NCBI Protein veritabanında Türkiye kaynaklı Capsicum annuum kayıt sayısı: 1
Türkiye kaynaklı 1 protein dizisi '/content/kapsaisin_genleri/izole_hatay_seti/NCBI_Turkiye_Biber_Proteinleri.faa' dosyasına kaydedildi.

 - WZN32304.1: WZN32304.1 ribulose-1,5-bisphosphate carboxylase/oxygenase large subunit, partial (chlorop... (127 aa)


## 6. ?zole Edilen 127 Amino Asitlik T?rkiye Kayd?n?n (`WZN32304.1`) ?ncelenmesi ve FASTA ??kt?s?
Sorgu sonucunda elde edilen lokal protein dizisi saf amino asit ve FASTA format?nda d??a aktar?l?r.


In [6]:
from Bio import SeqIO
import glob
import os

ARAMA_KLASORU = "/content/kapsaisin_genleri"
bulunan_kayitlar = []

# Klasördeki ve alt klasörlerdeki tüm .faa / .fasta dosyalarını tara
tum_dosyalar = glob.glob(f"{ARAMA_KLASORU}/**/*.faa", recursive=True) + \
               glob.glob(f"{ARAMA_KLASORU}/**/*.fasta", recursive=True)

seen_ids = set()

for dosya in tum_dosyalar:
    for record in SeqIO.parse(dosya, "fasta"):
        if len(record.seq) == 127 and record.id not in seen_ids:
            seen_ids.add(record.id)
            bulunan_kayitlar.append((dosya, record))

# Eğer önceki hücredeki 'tr_proteinler' veya 'cekirdek_kayitlar' bellekteyse ve 1 elemanlıysa onu da kontrol et
if not bulunan_kayitlar and 'tr_proteinler' in locals() and len(tr_proteinler) == 1:
    bulunan_kayitlar.append(("Bellek (tr_proteinler)", tr_proteinler[0]))

if bulunan_kayitlar:
    for kaynak, rec in bulunan_kayitlar:
        print("=" * 70)
        print(f"KAYNAK DOSYA : {kaynak}")
        print(f"PROTEİN ID   : {rec.id}")
        print(f"AÇIKLAMA     : {rec.description}")
        print(f"UZUNLUK      : {len(rec.seq)} aa")
        print("=" * 70)

        print("\n--- 1. SAF AMİNO ASİT DİZİSİ (Kopyalamaya Uygun) ---")
        print(str(rec.seq))

        print("\n--- 2. FASTA FORMATI ---")
        print(f">{rec.description}\n{rec.seq}")

        # Ayrı bir dosyaya kaydet
        cikti_dosyasi = "/content/127aa_izole_biber_proteini.fasta"
        SeqIO.write([rec], cikti_dosyasi, "fasta")
        print(f"\n[BİLGİ] Dizi '{cikti_dosyasi}' olarak kaydedildi.")
else:
    print("127 aa uzunluğunda dizi bulunamadı. Son oluşan listedeki tüm diziler yazdırılıyor:\n")
    for dosya in tum_dosyalar:
        kayitlar = list(SeqIO.parse(dosya, "fasta"))
        if len(kayitlar) == 1:
            rec = kayitlar[0]
            print(f">{rec.description} ({len(rec.seq)} aa)\n{rec.seq}\n")

KAYNAK DOSYA : /content/kapsaisin_genleri/izole_hatay_seti/NCBI_Turkiye_Biber_Proteinleri.faa
PROTEİN ID   : WZN32304.1
AÇIKLAMA     : WZN32304.1 ribulose-1,5-bisphosphate carboxylase/oxygenase large subunit, partial (chloroplast) [Capsicum annuum]
UZUNLUK      : 127 aa

--- 1. SAF AMİNO ASİT DİZİSİ (Kopyalamaya Uygun) ---
LTSLDRYKGRCYRIERVVGEKDQYIAYVAYPLDLFEEGSVTNMFTSIVGNVFGFKALRALRLEDLRVPTAYIKTFQGPPHGIQVERDKLNKYGRPLLGCTIKPKLGLSAKNYGRAVYECLRGGLDFT

--- 2. FASTA FORMATI ---
>WZN32304.1 ribulose-1,5-bisphosphate carboxylase/oxygenase large subunit, partial (chloroplast) [Capsicum annuum]
LTSLDRYKGRCYRIERVVGEKDQYIAYVAYPLDLFEEGSVTNMFTSIVGNVFGFKALRALRLEDLRVPTAYIKTFQGPPHGIQVERDKLNKYGRPLLGCTIKPKLGLSAKNYGRAVYECLRGGLDFT

[BİLGİ] Dizi '/content/127aa_izole_biber_proteini.fasta' olarak kaydedildi.


## 7. ColabFold / AlphaFold2 3B Yap? Tahmini ??in Toplu Girdi (`CSV` & `FASTA`) Haz?rl???
Kapsaisin yola??ndaki 5 hedef enzimin (`PAL`, `C4H`, `C3H`, `CAOMT`, `CS_AT3`) yerel FASTA dosyas?ndan (veya UniProt referanslar?ndan) okunmas?, n?kleotid dizisi ise 3 okuma ?er?evesinde (ORF) proteine ?evrilmesi (`translate_cds`) ve **ColabFold Batch** format?nda (`colabfold_batch_input.csv`) kaydedilmesi sa?lan?r.


In [7]:
import os
import re
import csv
import json
import urllib.request

# 1. Kapsaisin Biyosentez Yolağı Hedef Enzim Tanımları ve Filtre Desenleri
TARGET_ENZYMES = {
    "PAL": {
        "tr_name": "Fenilalanin Amonyak Liyaz",
        "uniprot_id": "B5LAW0",
        "patterns": [r"\bPAL\d*\b", r"phenylalanine\s+ammonia[\s\-]?lyase"],
    },
    "C4H": {
        "tr_name": "Sinamik Asit-4-Hidroksilaz",
        "uniprot_id": "Q9FQY6",
        "patterns": [r"\bC4H\d*\b", r"\bCYP73A\w*\b", r"cinnamate\s+4[\s\-]?monooxygenase", r"cinnamic\s+acid\s+4[\s\-]?hydroxylase"],
    },
    "C3H": {
        "tr_name": "p-Kumarik Asit-3-Hidroksilaz",
        "uniprot_id": "B5LAT7",
        "patterns": [r"\bC3H\d*\b", r"\bCYP98A\w*\b", r"coumarate\s+3[\s\-]?hydroxylase", r"coumaroyl.*3'?[\s\-]?monooxygenase"],
    },
    "CAOMT": {
        "tr_name": "Kafeik Asit O-Metiltransferaz",
        "uniprot_id": "Q9FQY8",
        "patterns": [r"\bCA?OMT\d*\b", r"caffeic\s+acid\s+(?:3[\s\-]?O[\s\-]?)?methyltransferase"],
    },
    "CS_AT3": {
        "tr_name": "Kapsaisin Sentaz / Asil-Transferaz (Pun1/AT3)",
        "uniprot_id": "D2Y3X2",
        "patterns": [r"\bPUN1\b", r"\bAT3\b", r"\bCSY1\b", r"capsaicin\s+synthase", r"acyltransferase\s+pun1"],
    },
}

CODON_TABLE = {
    'ATA':'I','ATC':'I','ATT':'I','ATG':'M','ACA':'T','ACC':'T','ACG':'T','ACT':'T',
    'AAC':'N','AAT':'N','AAA':'K','AAG':'K','AGC':'S','AGT':'S','AGA':'R','AGG':'R',
    'CTA':'L','CTC':'L','CTG':'L','CTT':'L','CCA':'P','CCC':'P','CCG':'P','CCT':'P',
    'CAC':'H','CAT':'H','CAA':'Q','CAG':'Q','CGA':'R','CGC':'R','CGG':'R','CGT':'R',
    'GTA':'V','GTC':'V','GTG':'V','GTT':'V','GCA':'A','GCC':'A','GCG':'A','GCT':'A',
    'GAC':'D','GAT':'D','GAA':'E','GAG':'E','GGA':'G','GGC':'G','GGG':'G','GGT':'G',
    'TCA':'S','TCC':'S','TCG':'S','TCT':'S','TTC':'F','TTT':'F','TTA':'L','TTG':'L',
    'TAC':'Y','TAT':'Y','TAA':'*','TAG':'*','TGC':'C','TGT':'C','TGA':'*','TGG':'W',
}

def is_nucleotide_seq(seq: str) -> bool:
    clean = re.sub(r"[^A-Za-z]", "", seq).upper()
    return bool(clean) and (sum(1 for c in clean if c in "ATGCNU") / len(clean)) > 0.90

def translate_cds(nt_seq: str) -> str:
    seq = re.sub(r"[^A-Za-z]", "", nt_seq).upper().replace("U", "T")
    best_prot = ""
    for frame in range(3):
        sub = seq[frame:]
        aa_list = []
        for i in range(0, len(sub) - 2, 3):
            aa = CODON_TABLE.get(sub[i:i+3], "X")
            if aa == "*":
                break
            aa_list.append(aa)
        cand = "".join(aa_list)
        if len(cand) > len(best_prot):
            best_prot = cand
    return best_prot

def parse_fasta(fasta_path: str):
    records, header, chunks = [], None, []
    with open(fasta_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(chunks)))
                header, chunks = line[1:].strip(), []
            else:
                chunks.append(re.sub(r"\s+", "", line))
        if header is not None:
            records.append((header, "".join(chunks)))
    return records

def classify_header(header: str):
    for key, info in TARGET_ENZYMES.items():
        if any(re.search(pat, header, flags=re.IGNORECASE) for pat in info["patterns"]):
            return key
    return None

def save_outputs(extracted, output_dir="samandag_kapsaisin_genleri", prefix="Samandag"):
    os.makedirs(output_dir, exist_ok=True)
    comb_fasta = os.path.join(output_dir, "all_5_capsaicin_enzymes.fasta")
    batch_csv = os.path.join(output_dir, "colabfold_batch_input.csv")

    with open(comb_fasta, "w", encoding="utf-8") as cf, \
         open(batch_csv, "w", newline="", encoding="utf-8") as csvf:
        writer = csv.writer(csvf)
        writer.writerow(["id", "sequence"])

        for key, hits in extracted.items():
            if not hits:
                continue
            single_path = os.path.join(output_dir, f"{prefix}_{key}.fasta")
            with open(single_path, "w", encoding="utf-8") as sf:
                for i, (hdr, seq) in enumerate(hits, 1):
                    job_id = f"{prefix}_{key}" if len(hits) == 1 else f"{prefix}_{key}_{i}"
                    sf.write(f">{job_id} | {hdr}\n{seq}\n")
                    cf.write(f">{job_id} | {hdr}\n{seq}\n")
                    writer.writerow([job_id, seq])
            print(f"[AYRILDI] {key:6s} -> {single_path} ({len(hits)} dizi)")

    print(f"\nColabFold Batch girdi dosyası hazır: {batch_csv}")

# --- ÇALIŞTIRMA ---
# Kendi Samandağ biberi FASTA dosyanız varsa dosya yolunu yazın, yoksa None bırakın:
LOCAL_FASTA_FILE = None  # Örn: "/content/samandag_biber_genler.fasta"

extracted_genes = {k: [] for k in TARGET_ENZYMES}

if LOCAL_FASTA_FILE and os.path.exists(LOCAL_FASTA_FILE):
    for hdr, seq in parse_fasta(LOCAL_FASTA_FILE):
        enz = classify_header(hdr)
        if enz:
            if is_nucleotide_seq(seq):
                seq = translate_cds(seq)
            extracted_genes[enz].append((hdr, seq))
    save_outputs(extracted_genes, prefix="Samandag")
else:
    for key, info in TARGET_ENZYMES.items():
        url = f"https://rest.uniprot.org/uniprotkb/{info['uniprot_id']}.fasta"
        lines = urllib.request.urlopen(url).read().decode("utf-8").splitlines()
        extracted_genes[key].append((lines[0][1:], "".join(lines[1:])))
    save_outputs(extracted_genes, prefix="Capsicum_annuum")

[AYRILDI] PAL    -> samandag_kapsaisin_genleri/Capsicum_annuum_PAL.fasta (1 dizi)
[AYRILDI] C4H    -> samandag_kapsaisin_genleri/Capsicum_annuum_C4H.fasta (1 dizi)
[AYRILDI] C3H    -> samandag_kapsaisin_genleri/Capsicum_annuum_C3H.fasta (1 dizi)
[AYRILDI] CAOMT  -> samandag_kapsaisin_genleri/Capsicum_annuum_CAOMT.fasta (1 dizi)
[AYRILDI] CS_AT3 -> samandag_kapsaisin_genleri/Capsicum_annuum_CS_AT3.fasta (1 dizi)

ColabFold Batch girdi dosyası hazır: samandag_kapsaisin_genleri/colabfold_batch_input.csv
